# 🌍 Notebook 1: Data Exploration & EDA
**Project:** A Hybrid Deep Learning Approach for Modelling Global CO₂ Emissions  
**Author:** Hafiza Alishba Naaz | NUST Islamabad 2026  
**Supervisor:** Dr. Tahir Mehmood

## 1.1 Import Libraries

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, ElasticNet, LassoCV, ElasticNetCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings('ignore')
print('✅ Libraries imported successfully!')

## 1.2 Load Dataset
> **Dataset:** Global Data on Sustainable Energy (Kaggle, 2023)  
> Download from: https://www.kaggle.com/datasets/anshtanwar/global-data-on-sustainable-energy  
> Place the CSV in the `data/` folder.

In [ ]:
df = pd.read_csv('../data/global-data-on-sustainable-energy.csv')
print(f'Dataset shape: {df.shape}')
print(f'Year range: {df["Year"].min()} - {df["Year"].max()}')
print(f'Unique countries: {df["Entity"].nunique()}')
df.head()

## 1.3 Dataset Overview

In [ ]:
print('Column list:')
for i, col in enumerate(df.columns):
    print(f'  {i:2}. {col}')

print('\nMissing values:')
print(df.isnull().sum().to_string())

## 1.4 Target Variable: CO₂ Emissions Distribution

In [ ]:
y_col = 'Value_co2_emissions_kt_by_country'

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Original distribution
df[y_col].dropna().plot(kind='hist', bins=60, ax=axes[0], color='#378ADD', edgecolor='white')
axes[0].set_title('CO₂ Emissions — Original (skewed)'); axes[0].set_xlabel('kt CO₂')

# Log-transformed
np.log1p(df[y_col].dropna()).plot(kind='hist', bins=60, ax=axes[1], color='#1D9E75', edgecolor='white')
axes[1].set_title('CO₂ Emissions — log1p (normalized)'); axes[1].set_xlabel('log1p(kt CO₂)')

# Boxplot per year
df.boxplot(column=y_col, by='Year', ax=axes[2], flierprops=dict(marker='.', markersize=2))
axes[2].set_title('CO₂ by Year'); axes[2].set_xlabel('Year')

plt.suptitle('')
plt.tight_layout()
plt.savefig('../results/target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(df[y_col].describe())

## 1.5 Correlation Heatmap

In [ ]:
# Fix Density column
df['Density\n(P/Km2)'] = pd.to_numeric(df['Density\n(P/Km2)'], errors='coerce')

numeric_cols = df.select_dtypes(include=[float, int]).columns.tolist()
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlBu_r', center=0, linewidths=0.3, annot_kws={'size': 7})
plt.title('Feature Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.savefig('../results/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.6 Select Top 10 CO₂ Emitting Countries

In [ ]:
target_col = 'Value_co2_emissions_kt_by_country'
df_energy = df.copy()

top_10_entities = (
    df.groupby('Entity')[target_col]
    .mean()
    .nlargest(10)
    .index
    .tolist()
)

print(f'Top 10 CO₂ Emitters:')
for i, c in enumerate(top_10_entities, 1):
    print(f'  {i}. {c}')

df_top10 = df[df['Entity'].isin(top_10_entities)].copy()
df_top10 = df_top10.sort_values(by=['Entity', 'Year'])
df_top10.to_csv('../data/top_10_emitters.csv', index=False)
print(f'\n✅ Saved top_10_emitters.csv — shape: {df_top10.shape}')

## 1.7 CO₂ Time Series — All 10 Countries

In [ ]:
top10_countries = top_10_entities
colors_ts = ['#378ADD','#1D9E75','#D85A30','#7F77DD','#BA7517',
             '#533AB7','#0F6E56','#993C1D','#639922','#3C3489']

fig, axes = plt.subplots(2, 5, figsize=(20, 7), sharey=False)
for ax, country, color in zip(axes.flatten(), top10_countries, colors_ts):
    sub = df_energy[df_energy['Entity'] == country][['Year', target_col]].dropna().sort_values('Year')
    ax.plot(sub['Year'], sub[target_col], marker='o', markersize=4, linewidth=2, color=color)
    ax.set_title(country, fontsize=10, fontweight='bold')
    ax.set_xlabel('Year', fontsize=8); ax.set_ylabel('kt CO₂', fontsize=8)
    ax.tick_params(labelsize=7); ax.grid(True, alpha=0.3)

plt.suptitle('CO₂ Emissions Time Series — Top 10 Emitters (2000–2019)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../results/co2_time_series_all.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.8 Feature Selection (Lasso / ElasticNet / Random Forest)

In [ ]:
drop_cols = ['Entity', 'Year']
feature_cols = [c for c in df.columns if c not in drop_cols + [target_col]]

X_raw = df[feature_cols].copy()
y_raw = df[target_col].copy()
valid_idx = y_raw.notna()
X_valid = X_raw[valid_idx].reset_index(drop=True)
y_valid = y_raw[valid_idx].reset_index(drop=True)

for col in X_valid.columns:
    X_valid[col] = pd.to_numeric(X_valid[col], errors='coerce')

y_log = np.log1p(y_valid)
X_tr, X_te, y_tr, y_te = train_test_split(X_valid, y_log, test_size=0.2, random_state=42)

imputer = SimpleImputer(strategy='median')
X_tr_imp = imputer.fit_transform(X_tr)
X_te_imp = imputer.transform(X_te)

scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_tr_imp)
X_te_sc = scaler.transform(X_te_imp)

# Random Forest feature importance
rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
rf.fit(X_tr_sc, y_tr)
fi = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(10, 7))
fi.plot(kind='barh', color=['#378ADD' if i < 5 else '#B5D4F4' for i in range(len(fi))])
plt.title('Feature Importance — Random Forest'); plt.xlabel('Importance Score')
plt.gca().invert_yaxis(); plt.tight_layout()
plt.savefig('../results/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop features (importance >= 2%):')
print(fi[fi >= 0.02].index.tolist())